# Prepare the Biogen public ADME dataset

This notebook loads the published Biogen dataset, converts its `log10(value)` endpoints back to their reported physical units with `10 ** value`, adds provenance, and writes category-specific files under `dataset/ADMET/curated/`. The original CSV is not modified.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


def find_repository_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the ChemFlow repository root.")


REPOSITORY_ROOT = find_repository_root()
INPUT_PATH = (
    REPOSITORY_ROOT
    / "dataset"
    / "ADMET"
    / "Biogen_ADME"
    / "Biogen_ADME.csv"
)
CURATED_ROOT = REPOSITORY_ROOT / "dataset" / "ADMET" / "curated"

biogen_df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(biogen_df):,} molecules from {INPUT_PATH}")

Loaded 3,521 molecules from /Users/liuy48/Desktop/ChemFlow/dataset/ADMET/Biogen_ADME/Biogen_ADME.csv


In [2]:
biogen_df

,Internal ID,Vendor ID,SMILES,CollectionName,LOG HLM_CLint (mL/min/kg),LOG MDR1-MDCK ER (B-A/A-B),LOG SOLUBILITY PH 6.8 (ug/mL),LOG PLASMA PROTEIN BINDING (HUMAN) (% unbound),LOG PLASMA PROTEIN BINDING (RAT) (% unbound),LOG RLM_CLint (mL/min/kg),...,OUTLIER_LOG PLASMA PROTEIN BINDING (HUMAN) (% unbound),OUTLIER_LOG PLASMA PROTEIN BINDING (RAT) (% unbound),OUTLIER_LOG MDR1-MDCK ER (B-A/A-B),OUTLIER_LOG SOLUBILITY PH 6.8 (ug/mL),AC_LOG HLM_CLint (mL/min/kg),AC_LOG RLM_CLint (mL/min/kg),AC_LOG PLASMA PROTEIN BINDING (HUMAN) (% unbound),AC_LOG PLASMA PROTEIN BINDING (RAT) (% unbound),AC_LOG MDR1-MDCK ER (B-A/A-B),AC_LOG SOLUBILITY PH 6.8 (ug/mL)
0,Mol1,317714313,CNc1cc(Nc2cccn(-c3ccccn3)c2=O)nn2c(C(=O)N[C@@H...,emolecules,0.675687,1.493167,0.089905,0.991226,0.518514,1.392169,...,False,False,False,False,False,False,False,False,False,False
1,Mol2,324056965,CCOc1cc2nn(CCC(C)(C)O)cc2cc1NC(=O)c1cccc(C(F)F)n1,emolecules,0.675687,1.040780,0.550228,0.099681,0.268344,1.027920,...,False,False,False,False,False,False,False,False,False,False
2,Mol3,304005766,CN(c1ncc(F)cn1)[C@H]1CCCNC1,emolecules,0.675687,-0.358806,NaN,2.000000,2.000000,1.027920,...,False,False,False,False,False,False,False,False,False,False
3,Mol4,194963090,CC(C)(Oc1ccc(-c2cnc(N)c(-c3ccc(Cl)cc3)c2)cc1)C...,emolecules,0.675687,1.026662,1.657056,-1.158015,-1.403403,1.027920,...,False,False,False,False,False,False,False,False,False,False
4,Mol5,324059015,CC(C)(O)CCn1cc2cc(NC(=O)c3cccc(C(F)(F)F)n3)c(C...,emolecules,0.996380,1.010597,NaN,1.015611,1.092264,1.629093,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3516,Mol3517,43258693,O=C(c1ccc2c(c1)CCCC2)N1CCOCC1c1ccn[nH]1,emolecules,NaN,0.606813,NaN,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,False,False
3517,Mol3518,27448206,O=C(Nc1nc2ccccc2[nH]1)c1ccc(-n2cccc2)cc1,emolecules,NaN,-0.444495,NaN,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,False,False
3518,Mol3519,207150215,NC(=O)c1noc([C@@H](CCCC2CCCCC2)CC(=O)NO)n1,emolecules,0.863799,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,False,False
3519,Mol3520,25037224,CCCCCCCCc1ccc(CC[C@](N)(CO)COP(=O)(O)O)cc1,emolecules,0.881385,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,False,False


In [3]:
biogen_df.columns

Index(['Internal ID', 'Vendor ID', 'SMILES', 'CollectionName',
       'LOG HLM_CLint (mL/min/kg)', 'LOG MDR1-MDCK ER (B-A/A-B)',
       'LOG SOLUBILITY PH 6.8 (ug/mL)',
       'LOG PLASMA PROTEIN BINDING (HUMAN) (% unbound)',
       'LOG PLASMA PROTEIN BINDING (RAT) (% unbound)',
       'LOG RLM_CLint (mL/min/kg)', 'MOL_smiles', 'MOL_molhash_id',
       'MOL_molhash_id_no_stereo', 'MOL_num_stereoisomers',
       'MOL_num_undefined_stereoisomers', 'MOL_num_defined_stereo_center',
       'MOL_num_undefined_stereo_center', 'MOL_num_stereo_center',
       'MOL_undefined_E_D', 'MOL_undefined_E/Z',
       'OUTLIER_LOG HLM_CLint (mL/min/kg)',
       'OUTLIER_LOG RLM_CLint (mL/min/kg)',
       'OUTLIER_LOG PLASMA PROTEIN BINDING (HUMAN) (% unbound)',
       'OUTLIER_LOG PLASMA PROTEIN BINDING (RAT) (% unbound)',
       'OUTLIER_LOG MDR1-MDCK ER (B-A/A-B)',
       'OUTLIER_LOG SOLUBILITY PH 6.8 (ug/mL)', 'AC_LOG HLM_CLint (mL/min/kg)',
       'AC_LOG RLM_CLint (mL/min/kg)',
       'AC_LOG PLASM

In [ ]:
LOG_TO_RAW_COLUMNS = {
    "LOG HLM_CLint (mL/min/kg)": "HLM_CLint (mL/min/kg)",
    "LOG RLM_CLint (mL/min/kg)": "RLM_CLint (mL/min/kg)",
    "LOG MDR1-MDCK ER (B-A/A-B)": "MDR1_MDCK_ER",
    "LOG SOLUBILITY PH 6.8 (ug/mL)": "Solubility_PH_6_8 (ug/mL)",
    "LOG PLASMA PROTEIN BINDING (HUMAN) (% unbound)": "Human_PPB (% Unbound)",
    "LOG PLASMA PROTEIN BINDING (RAT) (% unbound)": "Rat_PPB (% Unbound)",
}

missing_columns = sorted(set(LOG_TO_RAW_COLUMNS).difference(biogen_df.columns))
if missing_columns:
    raise KeyError(f"Biogen dataset is missing required columns: {missing_columns}")

converted_df = biogen_df[["Internal ID", "Vendor ID", "SMILES", "CollectionName"]].copy()
for log_column, raw_column in LOG_TO_RAW_COLUMNS.items():
    converted_df[raw_column] = np.power(10.0, biogen_df[log_column])

converted_df["source"] = "Biogen_ADME_Fang_2023"

for column in ["Human_PPB (% Unbound)", "Rat_PPB (% Unbound)"]:
    invalid = converted_df[column].notna() & ~converted_df[column].between(0, 100)
    if invalid.any():
        raise ValueError(f"{column} contains values outside 0–100%.")

converted_df.head()

In [ ]:
CATEGORY_COLUMNS = {
    "physchem": [
        "Solubility_PH_6_8 (ug/mL)",
    ],
    "clearance": [
        "HLM_CLint (mL/min/kg)",
        "RLM_CLint (mL/min/kg)",
    ],
    "permeability": [
        "MDR1_MDCK_ER",
    ],
    "protein_binding": [
        "Human_PPB (% Unbound)",
        "Rat_PPB (% Unbound)",
    ],
}
COMMON_COLUMNS = ["Internal ID", "Vendor ID", "SMILES", "CollectionName", "source"]

export_summary = []
for category, endpoint_columns in CATEGORY_COLUMNS.items():
    category_df = (
        converted_df.loc[:, COMMON_COLUMNS + endpoint_columns]
        .dropna(subset=endpoint_columns, how="all")
        .reset_index(drop=True)
    )
    output_directory = CURATED_ROOT / category
    output_directory.mkdir(parents=True, exist_ok=True)
    output_path = output_directory / f"Biogen_{category}.csv"
    category_df.to_csv(output_path, index=False)
    export_summary.append(
        {
            "category": category,
            "rows": len(category_df),
            "endpoints": ", ".join(endpoint_columns),
            "output": str(output_path),
        }
    )

export_summary_df = pd.DataFrame(export_summary)
export_summary_df